In [ ]:
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.gridspec as gridspec

import sys
sys.path.append('../data')
from utils.functions import load_yaml
from embviz import plot_driver_embedding, plot_am_embedding, plot_org_embedding,visualize_feat_with_am

In [ ]:
# Load Configuration
config = load_yaml("../config/RRGCL.yaml")
SAVEPATH = config.SAVEPATH

# Cancer Driver DataFrame
mut_df = pd.read_csv(f'{config.DATABASE}/matched_cancer_driver_df.csv')

In [ ]:
mut_df.label.value_counts()

In [ ]:
models = {
    "All": 'szusj1rl',
    'All+Egy': 'rtn2y7dv',
    # 'All+Egy2': 'di9woh2e',
    'ExLocalGeo': '3rki24k9',
    'ExLocalGeo+Egy': '3v9dkf3h',
    'ExSpEnv': 'v6iuy1du',
    'ExSpEnv+Egy': 'sryxn215',
    'ExEvol': "dytsoorb",
    'ExEvol+Egy': 'fa2h3zct'
}

# Cancer Driver Visualization

In [ ]:
plot_driver_embedding(SAVEPATH, models, method='umap', num_cols=2) # tsne, umap, pca

In [ ]:
plot_am_embedding(SAVEPATH, models, method='umap', target='avg_am', num_cols=2, portion=0.15, only_driver=True)

In [ ]:
plot_am_embedding(SAVEPATH, models, method='umap', target='max_am', num_cols=2, portion=0.15, only_driver=True)

# Original Feature Plotting

In [ ]:
sample_df = pd.read_csv(f"{SAVEPATH}/DGI/rtn2y7dv/tSNE_result_driver+am.csv")

vis_org_df = pd.read_csv(f'{SAVEPATH}/DGI/tSNE_result_all-feat_org.csv')
vis_org_df = vis_org_df.merge(mut_df[['node_id', 'label']], on='node_id', how='left')
vis_org_df = vis_org_df[vis_org_df['node_id'].isin(sample_df['node_id'])]

vis_org_df = vis_org_df[~vis_org_df['label_x'].isna()]

In [ ]:
vis_org_df

In [ ]:
# 6. Visualize by Label and max_am side-by-side in a 1x2 Subplot
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# [Left] Scatter plot by Label
sns.scatterplot(
    x='umap_1', y='umap_2', hue='label_x', palette='Set2',
    data=vis_org_df, alpha=0.7, ax=axes[0], s=100
)
axes[0].set_title('UMAP of Original Features by Label')
axes[0].set_xlabel('UMAP 1')
axes[0].set_ylabel('UMAP 2')

# [Right] Scatter plot by max_am
scatter = axes[1].scatter(
    vis_org_df['tsne_1'], vis_org_df['tsne_2'],
    c=vis_org_df['avg_am'], cmap='viridis_r', alpha=0.7,
    edgecolors='w', linewidth=0.2, s=100
)
axes[1].set_title('UMAP of Original Features by avg_am')
axes[1].set_xlabel('UMAP 1')
axes[1].set_ylabel('UMAP 2')

# Add colorbar
cbar = fig.colorbar(scatter, ax=axes[1])
cbar.set_label('Average AlphaMissense Score (avg_am)')
plt.tight_layout()
plt.show()

# Plot with Other Feature

In [ ]:
def check_mixed(x):
    if x.nunique() == 1:
        return x.iloc[0]
    else:
        return 'mixed'

In [ ]:
org_edge_df = pd.read_csv(f"{config.DATABASE}/step2_rmStrangeEdge_human_aa_edges_exclubq_Nucleosome_related_data_v031626.csv")
feat_df = org_edge_df[['nodeid_1', 'nodeid_2', 'pdb_code', 'source', 'chain_flag']]
        
result_df1 = feat_df.groupby('nodeid_1')[['source', 'chain_flag']].agg(check_mixed).reset_index()
result_df1.rename({'nodeid_1': 'node_id'}, inplace=True, axis=1)
result_df2 = feat_df.groupby('nodeid_2')[['source', 'chain_flag']].agg(check_mixed).reset_index()
result_df2.rename({'nodeid_2': 'node_id'}, inplace=True, axis=1)

final_result_df = pd.merge(result_df1, result_df2, on=['node_id', 'source', 'chain_flag'], how='inner')
cleaned_final_info = final_result_df[final_result_df['source'] != 'mixed']
cleaned_final_info.head(3)

In [ ]:
ref_df = pd.read_csv(f"{SAVEPATH}/DGI/szusj1rl/tSNE_result_driver+am.csv") # All
# ref_df = pd.read_csv(f"{SAVEPATH}/DGI/rtn2y7dv/tSNE_result_driver+am.csv") # All+Egy
# ref_df = pd.read_csv(f"{SAVEPATH}/DGI/3rki24k9/tSNE_result_driver+am.csv") # ExLocalGeo
# ref_df = pd.read_csv(f"{SAVEPATH}/DGI/3v9dkf3h/tSNE_result_driver+am.csv") # ExLocalGeo+Energy
# ref_df = pd.read_csv(f"{SAVEPATH}/DGI/sryxn215/tSNE_result_driver+am.csv") # ExSpENV+Energy
# ref_df = pd.read_csv(f"{SAVEPATH}/DGI/v6iuy1du/tSNE_result_driver+am.csv") # ExSpENV
# ref_df = pd.read_csv(f"{SAVEPATH}/DGI/dytsoorb/tSNE_result_driver+am.csv") # ExEvol
# ref_df = pd.read_csv(f"{SAVEPATH}/DGI/fa2h3zct/tSNE_result_driver+am.csv") # ExEvol+Energy

In [ ]:
tgt_feature = 'source'

plot_df = pd.merge(ref_df, cleaned_final_info, how='left', on='node_id')
plot_df = plot_df[(~plot_df[tgt_feature].isna())]

print("Original")
print(plot_df[tgt_feature].value_counts())

visualize_feat_with_am(plot_df, feature=tgt_feature,  # chain_flag or source
                       must_incl_val = 'mixed' if tgt_feature=='chain_flag' else 'core',
                       num_sample = 300 if tgt_feature=='chain_flag' else 1000,
                       point_size=100, 
                       palette=["#FFBF00", "Navy"]
                       )

In [ ]:
tgt_feature = 'chain_flag'

plot_df = pd.merge(ref_df, cleaned_final_info, how='left', on='node_id')
plot_df = plot_df[(~plot_df[tgt_feature].isna())]

print("Original")
print(plot_df[tgt_feature].value_counts())

visualize_feat_with_am(plot_df, feature=tgt_feature,  # chain_flag or source
                       must_incl_val = 'mixed' if tgt_feature=='chain_flag' else 'core',
                       num_sample = 300 if tgt_feature=='chain_flag' else 1000,
                       point_size=100, 
                       palette=["Darkred", "Skyblue"]
                       )